# Connecting Python to Claude API

## Learning objectives
By the end of this notebook, you will be able to:

1. Explain the relationship between an application, the Anthropic SDK, and Claude.
2. Configure an Anthropic API key safely using environment variables.
3. Send a message to Claude using the official Python SDK.
4. inspect the response, stop reason, and token usage.
5. Handle common API errors and understand basic cost controls.

> **Prerequisites:** Python 3.10+, an Anthropic Console account, an API key, and API billing/credits enabled. API access is separate from a Claude web-chat subscription.

## Architecture

`Python application → Anthropic Python SDK → Claude Messages API → Claude model → response`

This notebook uses the direct Anthropic SDK. Notebook 02 will introduce LangChain's standardized chat-model interface.

## 1. Environment setup

If you are using the course Poetry project, run these commands in a terminal (not in a notebook cell):

```bash
poetry add anthropic python-dotenv
```

If you are using a regular virtual environment instead:

```bash
python -m pip install anthropic python-dotenv
```

Select the `Python (LangGraph Module 2)` kernel in Jupyter if you created it.

In [ ]:
# Verify the Python version and install visibility.
import sys
print("Python:", sys.version)

## 2. Store the API key securely

Create a file named `.env` in the project root (the same folder from which you launch Jupyter):

```text
ANTHROPIC_API_KEY=your_actual_api_key
CLAUDE_MODEL=your_model_id
ANTHROPIC_WORKSPACE_ID=your_workspace_id
```

`ANTHROPIC_WORKSPACE_ID` is only required if your API key is not scoped to a workspace. Find the workspace ID in the Anthropic Console; do not use the workspace name. If your key is already scoped to a workspace, you can omit this setting.

Get the model ID from the Anthropic Console or current model documentation:
https://platform.claude.com/docs/en/about-claude/models/overview

Do not paste your API key into a notebook cell, commit `.env` to Git, or include it in screenshots.

Create a `.gitignore` file containing:

```text
.env
.venv/
__pycache__/
.ipynb_checkpoints/
```

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # Reads .env from the current working directory / parent directories.

api_key = os.getenv("ANTHROPIC_API_KEY")
model = os.getenv("CLAUDE_MODEL")
workspace_id = os.getenv("ANTHROPIC_WORKSPACE_ID")

if not api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY was not found. Add it to your .env file "
        "and restart the notebook kernel."
    )

if not model:
    raise RuntimeError(
        "CLAUDE_MODEL was not found. Set it to a model ID available to your API account."
    )

if not workspace_id:
    raise RuntimeError(
        "ANTHROPIC_WORKSPACE_ID was not found. Add it to your .env file "
        "and restart the notebook kernel."
    )
    
print("API key loaded:", bool(api_key))
print("Configured model:", model)
print("Workspace ID loaded:", bool(workspace_id))

API key loaded: True
Configured model: claude-haiku-5-5
Workspace ID loaded: True


## 3. Create an Anthropic client

The SDK client handles authentication and communication with the Messages API. Keep the key in an environment variable; do not print it.

Documentation: https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python

In [2]:
from anthropic import Anthropic

client = Anthropic(
    api_key=api_key,
    default_headers=(
        {"anthropic-workspace-id": workspace_id}
        if workspace_id
        else None
    ),
)
print("Anthropic client created.")

Anthropic client created.


## 4. Make the first Claude API request

The Messages API accepts a sequence of messages. Here we send one user message and ask for a concise explanation.

Key parameters:

- `model`: the model identifier enabled for your API account.
- `max_tokens`: the maximum number of output tokens to generate.
- `messages`: conversation messages, each with a role and content.

In [3]:
response = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": "Explain what an AI agent is in simple terms. Give one business example."
        }
    ]
)

for block in response.content:
    if block.type == "text":
        print(block.text)

An AI agent is a software program that uses artificial intelligence to pursue a goal on its own. Instead of just answering one question and stopping, it can plan a series of steps, use tools (like a web browser, a calendar, or a database), check its own progress, and adjust until the task is done. You give it an objective, and it figures out how to get there, usually with some level of human oversight.

**Business example:** A customer support AI agent for an online retailer. A customer writes, "My order arrived damaged, I want a replacement." The agent looks up the order in the company's system, verifies the delivery date and return policy, checks inventory, creates a replacement order, emails the customer a confirmation with tracking details, and logs the case for the warehouse team. If something falls outside its rules, such as a refund over a set amount, it hands the case to a human employee.


In [4]:
response

Message(id='msg_011Ch2TrS6pJTxRkwpnb4pZ9', container=None, content=[TextBlock(citations=None, text='An AI agent is a software program that uses artificial intelligence to pursue a goal on its own. Instead of just answering one question and stopping, it can plan a series of steps, use tools (like a web browser, a calendar, or a database), check its own progress, and adjust until the task is done. You give it an objective, and it figures out how to get there, usually with some level of human oversight.\n\n**Business example:** A customer support AI agent for an online retailer. A customer writes, "My order arrived damaged, I want a replacement." The agent looks up the order in the company\'s system, verifies the delivery date and return policy, checks inventory, creates a replacement order, emails the customer a confirmation with tracking details, and logs the case for the warehouse team. If something falls outside its rules, such as a refund over a set amount, it hands the case to a hum

## 5. Inspect the response

The response contains more than generated text. Inspect its ID, model, stop reason, and usage metadata. Token counts are useful for monitoring usage and estimating cost; exact pricing depends on the model and current provider pricing.

In [5]:
print("Response ID:", response.id)
print("Model returned:", response.model)
print("Stop reason:", response.stop_reason)
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)

Response ID: msg_011Ch2TrS6pJTxRkwpnb4pZ9
Model returned: claude-haiku-5-5
Stop reason: end_turn
Input tokens: 30
Output tokens: 266


## 6. Use a reusable function

Applications should avoid repeating request code. Wrap the API call in a small function and keep model configuration in one place.

In [ ]:
def ask_claude(prompt: str, max_tokens: int = 400) -> str:
    """Send one user prompt to Claude and return concatenated text blocks."""
    result = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )

    text_parts = [
        block.text for block in result.content
        if block.type == "text"
    ]
    return "\n".join(text_parts)


answer = ask_claude(
    "List three ways a data analyst can use an AI assistant.",
    max_tokens=250,
)
print(answer)

## 7. Conversation history

The Messages API is stateless at the request level: to continue a conversation, send the relevant previous messages again. The application is responsible for retaining conversation history.

This simple example sends two turns in one request.

In [ ]:
conversation = [
    {"role": "user", "content": "I manage sales data for a retail company."},
    {"role": "assistant", "content": "Understood. What would you like to analyze?"},
    {"role": "user", "content": "Suggest three useful KPIs for monthly sales performance."},
]

chat_response = client.messages.create(
    model=model,
    max_tokens=300,
    messages=conversation,
)

for block in chat_response.content:
    if block.type == "text":
        print(block.text)

## 8. Handle common API errors

Common issues include invalid credentials, insufficient API credits, rate limits, network failures, and unsupported model IDs. Do not expose credentials or full sensitive request payloads in error logs.

The SDK provides typed exceptions. This cell demonstrates a safe, minimal pattern.

In [ ]:
from anthropic import (
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
    APIStatusError,
)

def safe_claude_call(prompt: str) -> str:
    try:
        result = client.messages.create(
            model=model,
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}],
        )
        return "\n".join(
            block.text for block in result.content
            if block.type == "text"
        )
    except AuthenticationError:
        return "Authentication failed. Check the API key configured in .env."
    except RateLimitError:
        return "Rate limit reached. Wait briefly and retry with backoff."
    except APIConnectionError:
        return "Could not connect to the API. Check network connectivity."
    except APIStatusError as exc:
        # Status code is useful for diagnosis; avoid printing secrets or payloads.
        return f"API returned an error (HTTP {exc.status_code}). Check model access and billing."

print(safe_claude_call("In one sentence, define retrieval-augmented generation."))

## 9. Optional: Stream the response

Streaming displays generated text as it arrives instead of waiting for the complete response. This is useful for interactive applications.

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=250,
    messages=[
        {"role": "user", "content": "Explain the difference between an LLM and an AI agent."}
    ],
) as stream:
    for text_chunk in stream.text_stream:
        print(text_chunk, end="", flush=True)
print()

## 10. Mini exercise

**Task:** Ask Claude to act as a business analyst and explain the following metrics to a non-technical manager:

- Revenue: ₹12,50,000
- Cost: ₹8,00,000
- Previous-month revenue: ₹10,00,000

Ask it to:
1. Explain revenue growth in plain language.
2. Explain why profit and revenue are different.
3. State any calculation assumptions.

**Important:** In this exercise Claude is only explaining supplied values. In later lessons, Python tools will perform deterministic calculations.

In [ ]:
exercise_prompt = """
You are a business analyst communicating with a non-technical manager.

Given:
- Current-month revenue: INR 1,250,000
- Current-month cost: INR 800,000
- Previous-month revenue: INR 1,000,000

Explain:
1. Revenue growth from the previous month.
2. The difference between revenue and profit.
3. Any assumptions you make.

Use clear, concise business language.
"""

print(ask_claude(exercise_prompt, max_tokens=350))

## 11. Knowledge check

1. Why should an API key be stored in an environment variable?
2. What is the purpose of `max_tokens`?
3. Why must an application send prior messages again to continue a conversation?
4. What is the difference between input tokens and output tokens?
5. Why should financial calculations eventually be delegated to deterministic Python functions?

<details>
<summary>Suggested answers</summary>

1. To reduce accidental exposure in source code, notebooks, and version control.
2. It limits the maximum number of generated output tokens for the request.
3. The Messages API does not automatically retain application conversation state between independent requests.
4. Input tokens represent content sent to the model; output tokens represent generated content.
5. Deterministic functions are reproducible, testable, and more reliable for exact arithmetic.
</details>

## Key takeaway

You have connected a Python application directly to Claude, sent messages, inspected token usage, reused a helper function, handled common errors, and tried streaming.

**Next:** Notebook 02 introduces LangChain's `ChatAnthropic` integration and the common chat-model interface.